In [1]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm
from pathlib import Path

# Load data

In [2]:
Xy_folderpath = Path('../data/silver/Xy-2021')

track_train_df = pd.read_parquet(Xy_folderpath / "track_train_df.parquet")
Xy_train = pd.read_parquet(Xy_folderpath / "Xy_train.parquet")
track_test_df = pd.read_parquet(Xy_folderpath / "track_test_df.parquet")
Xy_test = pd.read_parquet(Xy_folderpath / "Xy_test.parquet")

Xy_train.head(1)

,red,red_3x3_median,red_3x3_min,red_3x3_max,red_3x3_var,red_5x5_median,red_5x5_min,red_5x5_max,red_5x5_var,red_11x11_median,...,cloud_confidence_5x5_min,cloud_confidence_5x5_max,cloud_confidence_5x5_var,cloud_confidence_11x11_median,cloud_confidence_11x11_min,cloud_confidence_11x11_max,cloud_confidence_11x11_var,scene_classification,valid_observations,y
0,0.133333,0.133333,0.121569,0.141176,0.007843,0.133333,0.109804,0.145098,0.019608,0.12549,...,0,0,0,0,0,0,0,5,7,False


# Train models

In [3]:
# Build X,y
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Initialize and fit the Dummy Classifier
dummy_clf = DummyClassifier(strategy="stratified")
dummy_clf.fit(X_train, y_train > 0.65)
yhat_test = dummy_clf.predict(X_test)

# Evaluate the baseline
print(f"Baseline Macro F1: {f1_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")
print(f"Baseline Macro Precision: {precision_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")
print(f"Baseline Macro Recall: {recall_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")
print("- Negative (0)")
print(f"- F1: {f1_score(y_test, yhat_test, pos_label=0, zero_division=0):.4f}")
print(f"-- Precision: {precision_score(y_test, yhat_test, pos_label=0, zero_division=0):.4f}")
print(f"-- Recall: {recall_score(y_test, yhat_test, pos_label=0, zero_division=0):.4f}")
print("- Positive (1)")
print(f"- F1: {f1_score(y_test, yhat_test, pos_label=1, zero_division=0):.4f}")
print(f"-- Precision: {precision_score(y_test, yhat_test, pos_label=1, zero_division=0):.4f}")
print(f"-- Recall: {recall_score(y_test, yhat_test, pos_label=1, zero_division=0):.4f}")

Baseline Macro F1: 0.4713
Baseline Macro Precision: 0.4965
Baseline Macro Recall: 0.4968
- Negative (0)
- F1: 0.5110
-- Precision: 0.4225
-- Recall: 0.6465
- Positive (1)
- F1: 0.4316
-- Precision: 0.5705
-- Recall: 0.3470


# Prep hyperparam optimization

In [4]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

def optuna_objective_factory(X, y, groups):
    def objective(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 2, 15),
            'num_leaves': trial.suggest_int('num_leaves', 10, 256),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'n_jobs': -1,
            'verbose': -1,
            'random_state': 42
        }
        
        
        # Use cross-validation to evaluate the parameters on the training data
        model = lgb.LGBMClassifier(**param)
        cv = GroupKFold(n_splits=5)
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring='f1_macro', n_jobs=1)
        
        return scores.mean()

    return objective

# Train LGBM baseline

In [5]:
# Baseline LGBM
def get_base_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_features(X_train)
X_test = get_base_features(X_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608


In [6]:
def train_test_lgbm(X_train, y_train, X_test, y_test, groups, n_trials=3):
    print("Starting Optuna hyperparameter optimization...")
    study = optuna.create_study(direction="maximize") 
    study.optimize(optuna_objective_factory(X_train, y_train, groups), n_trials=n_trials, show_progress_bar=True)
    
    # Build final model
    clf = lgb.LGBMClassifier(**study.best_params)
    clf.fit(X_train, y_train)
    
    yhat_train = clf.predict(X_train)
    yhat_test = clf.predict(X_test)
    
    print("\n--- OPTIMIZED MODEL EVALUATION ---")
    print("TRAIN")
    print(f"- Macro F1: {f1_score(y_train, yhat_train, average='macro', zero_division=0):.4f}")
    print(f"- Macro Precision: {precision_score(y_train, yhat_train, average='macro', zero_division=0):.4f}")
    print(f"- Macro Recall: {recall_score(y_train, yhat_train, average='macro', zero_division=0):.4f}")
    
    print("\nTEST")
    print(f"- Macro F1: {f1_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")
    print(f"- Macro Precision: {precision_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")
    print(f"- Macro Recall: {recall_score(y_test, yhat_test, average='macro', zero_division=0):.4f}")

    return clf, yhat_train, yhat_test

train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:45:05,409] A new study created in memory with name: no-name-e217261f-5371-4189-b20e-1e9ad118ed95


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:45:06,696] Trial 0 finished with value: 0.8921720109231203 and parameters: {'n_estimators': 154, 'learning_rate': 0.07002177123928535, 'max_depth': 2, 'num_leaves': 236, 'subsample': 0.6880963473942587, 'colsample_bytree': 0.5192545630758412, 'min_child_samples': 89}. Best is trial 0 with value: 0.8921720109231203.
[I 2026-06-18 16:45:24,403] Trial 1 finished with value: 0.9031273800773894 and parameters: {'n_estimators': 281, 'learning_rate': 0.05840189154171083, 'max_depth': 14, 'num_leaves': 217, 'subsample': 0.6759277828771209, 'colsample_bytree': 0.8259140305899553, 'min_child_samples': 42}. Best is trial 1 with value: 0.9031273800773894.
[I 2026-06-18 16:45:27,388] Trial 2 finished with value: 0.8910809266954093 and parameters: {'n_estimators': 329, 'learning_rate': 0.021483045438973666, 'max_depth': 2, 'num_leaves': 32, 'subsample': 0.7015818893471193, 'colsample_bytree': 0.66494992079206, 'min_child_samples': 98}. Best is trial 1 with value: 0.9031273800773894.

# Consider the SCL

In [10]:
# The SCL flags a lot of px as cloud and whatnot, but not vegetation
SCL_IDX_TO_SCL_NAME = {
    0: 'NO_DATA',
    1: 'DEFECTIVE_PX',
    2: 'TOPO_SHADOW',
    3: 'CLOUD_SHADOW',
    4: 'VEGETATION',
    5: 'NOT_VEGETATED',
    6: 'WATER',
    7: 'UNCLASSIFIED',
    8: 'CLOUD_MED_PROB',
    9: 'CLOUD_HIGH_PROB',
    10: 'THIN_CIRRUS',
    11: 'SNOW_OR_ICE',
}

def add_scl(Xy):
    Xy['scl'] = Xy.scene_classification.map(SCL_IDX_TO_SCL_NAME).astype('category')

add_scl(Xy_train)
add_scl(Xy_test)

In [12]:
# Take into account this SCL
def get_base_and_scl_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl']
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_and_scl_features(X_train)
X_test = get_base_and_scl_features(X_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,scl
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608,NOT_VEGETATED


In [13]:
train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:47:22,512] A new study created in memory with name: no-name-44ba4353-f5ba-484c-8d1c-a555e73ef750


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:47:28,819] Trial 0 finished with value: 0.8998332936266676 and parameters: {'n_estimators': 100, 'learning_rate': 0.028636393684254503, 'max_depth': 8, 'num_leaves': 227, 'subsample': 0.76920024922479, 'colsample_bytree': 0.5651009256683444, 'min_child_samples': 15}. Best is trial 0 with value: 0.8998332936266676.
[I 2026-06-18 16:47:33,259] Trial 1 finished with value: 0.8977425555791084 and parameters: {'n_estimators': 349, 'learning_rate': 0.023221191956741667, 'max_depth': 4, 'num_leaves': 28, 'subsample': 0.6158902885766141, 'colsample_bytree': 0.5213515267334108, 'min_child_samples': 100}. Best is trial 0 with value: 0.8998332936266676.
[I 2026-06-18 16:47:41,039] Trial 2 finished with value: 0.9010886542737545 and parameters: {'n_estimators': 486, 'learning_rate': 0.022263485856101613, 'max_depth': 15, 'num_leaves': 19, 'subsample': 0.8149510879679114, 'colsample_bytree': 0.8350998500532518, 'min_child_samples': 81}. Best is trial 2 with value: 0.901088654273754

# Add TCI & other features

In [14]:
# https://clearsky.vision/knowledge/sentinel2-indices-cheatsheet

def compute_tci(df, suffix):
    return 1.2 * (df[f"rededge{suffix}"] - df[f"green{suffix}"]) - 1.5 * (df[f"red{suffix}"] - df[f"green{suffix}"]) * np.sqrt(df[f"rededge{suffix}"] / df[f"red{suffix}"])

def compute_ndvi(df, suffix):
    return (df[f"nir{suffix}"] - df[f"red{suffix}"]) / (df[f"nir{suffix}"] + df[f"red{suffix}"])

def compute_evi(df, suffix):
    G, C1, C2, L = 2.5, 6, 7.5, 1.0
    return G * (df[f"nir{suffix}"] - df[f"red{suffix}"]) / (df[f"nir{suffix}"] + C1 * df[f"red{suffix}"] + C2 * df[f"blue{suffix}"] + L)

def compute_evi2(df, suffix):
    return (df[f"nir{suffix}"] - df[f"swir11{suffix}"]) / (df[f"nir{suffix}"] + df[f"swir11{suffix}"])

def compute_ndre_b5(df, suffix):
    return (df[f"B8A{suffix}"] - df[f"rededge{suffix}"]) / (df[f"B8A{suffix}"] + df[f"rededge{suffix}"])

def compute_ndre_b6(df, suffix):
    return (df[f"B8A{suffix}"] - df[f"B06{suffix}"]) / (df[f"B8A{suffix}"] + df[f"B06{suffix}"])

def compute_ndre_b7(df, suffix):
    return (df[f"B8A{suffix}"] - df[f"B07{suffix}"]) / (df[f"B8A{suffix}"] + df[f"B07{suffix}"])

def compute_gndvi(df, suffix):
    return 2.5 * (df[f"nir{suffix}"] - df[f"red{suffix}"]) / (df[f"nir{suffix}"] + 2.4 * df[f"red{suffix}"] + 1) 

def compute_ndwi(df, suffix):
    return (df[f"nir{suffix}"] - df[f"swir11{suffix}"]) / (df[f"nir{suffix}"] + df[f"swir11{suffix}"])

def add_enhanced_features(df):
    df["tci"] = compute_tci(df, '').copy()
    df["ndvi"] = compute_ndvi(df, '').copy()
    df["evi"] = compute_evi(df, '').copy()
    df["evi2"] = compute_evi2(df, '').copy()
    df["ndre_b5"] = compute_ndre_b5(df, '').copy()
    df["ndre_b6"] = compute_ndre_b6(df, '').copy()
    df["ndre_b7"] = compute_ndre_b7(df, '').copy()
    df["gndvi"] = compute_gndvi(df, '').copy()
    df["ndwi"] = compute_ndwi(df, '').copy()

add_enhanced_features(Xy_train) 
add_enhanced_features(Xy_test) 

Xy_test.head(1)

,red,red_3x3_median,red_3x3_min,red_3x3_max,red_3x3_var,red_5x5_median,red_5x5_min,red_5x5_max,red_5x5_var,red_11x11_median,...,scl,tci,ndvi,evi,evi2,ndre_b5,ndre_b6,ndre_b7,gndvi,ndwi
93968,0.058824,0.058824,0.058824,0.058824,0.0,0.058824,0.054902,0.062745,0.0,0.058824,...,NOT_VEGETATED,0.036883,0.302326,0.07135,-0.176471,0.127273,0.033333,0.0,0.101881,-0.176471


In [16]:
# Take into account this SCL
def get_base_and_scl_and_advanced_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl']
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_and_scl_and_advanced_features(X_train)
X_test = get_base_and_scl_and_advanced_features(X_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,scl,tci,ndvi,evi,evi2,ndre_b5,ndre_b6,ndre_b7,gndvi,ndwi
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608,NOT_VEGETATED,0.03115,0.218391,0.074277,-0.116667,0.081633,0.039216,0.009524,0.12192,-0.116667


In [17]:
train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:48:03,578] A new study created in memory with name: no-name-dd21bc14-9280-4db7-90f2-45e33e1fc11f


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:48:31,996] Trial 0 finished with value: 0.9041620114807859 and parameters: {'n_estimators': 327, 'learning_rate': 0.019091734765047136, 'max_depth': 14, 'num_leaves': 197, 'subsample': 0.9970192862258596, 'colsample_bytree': 0.591353105229276, 'min_child_samples': 11}. Best is trial 0 with value: 0.9041620114807859.
[I 2026-06-18 16:48:46,452] Trial 1 finished with value: 0.9014923112109823 and parameters: {'n_estimators': 481, 'learning_rate': 0.015123548381654852, 'max_depth': 13, 'num_leaves': 34, 'subsample': 0.5349335961199229, 'colsample_bytree': 0.6673894001908542, 'min_child_samples': 22}. Best is trial 0 with value: 0.9041620114807859.
[I 2026-06-18 16:48:52,846] Trial 2 finished with value: 0.8985963929522043 and parameters: {'n_estimators': 178, 'learning_rate': 0.017135537618546987, 'max_depth': 7, 'num_leaves': 40, 'subsample': 0.5269950028039199, 'colsample_bytree': 0.9000182794248868, 'min_child_samples': 11}. Best is trial 0 with value: 0.90416201148078

# Keep track of #valid observations

In [21]:
# Take into account this SCL
def get_base_and_scl_and_obs_cnt_and_advanced_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl']
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    features += ['valid_observations']
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_and_scl_and_obs_cnt_and_advanced_features(X_train)
X_test = get_base_and_scl_and_obs_cnt_and_advanced_features(X_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,...,tci,ndvi,evi,evi2,ndre_b5,ndre_b6,ndre_b7,gndvi,ndwi,valid_observations
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608,...,0.03115,0.218391,0.074277,-0.116667,0.081633,0.039216,0.009524,0.12192,-0.116667,7


In [19]:
train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:49:00,589] A new study created in memory with name: no-name-b9f1746d-745a-44d6-bf3c-0e3fedecc518


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:49:08,271] Trial 0 finished with value: 0.9017392702594386 and parameters: {'n_estimators': 421, 'learning_rate': 0.03890220538975092, 'max_depth': 4, 'num_leaves': 235, 'subsample': 0.6655688032728468, 'colsample_bytree': 0.8438402040086054, 'min_child_samples': 59}. Best is trial 0 with value: 0.9017392702594386.
[I 2026-06-18 16:49:33,351] Trial 1 finished with value: 0.8990835495762882 and parameters: {'n_estimators': 473, 'learning_rate': 0.1362031433193414, 'max_depth': 10, 'num_leaves': 131, 'subsample': 0.5593688463676701, 'colsample_bytree': 0.9765291868078898, 'min_child_samples': 66}. Best is trial 0 with value: 0.9017392702594386.
[I 2026-06-18 16:50:06,779] Trial 2 finished with value: 0.9024245678791669 and parameters: {'n_estimators': 469, 'learning_rate': 0.01123803891224382, 'max_depth': 10, 'num_leaves': 128, 'subsample': 0.7900010372892216, 'colsample_bytree': 0.8442734664381172, 'min_child_samples': 25}. Best is trial 2 with value: 0.902424567879166

# Keep track of cloud confidence

In [22]:
# Take into account cloud confidence
def get_base_and_scl_and_obs_cnt_and_cloud_conf_and_advanced_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl']
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    features += ['valid_observations']
    features += ['cloud_confidence']
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_and_scl_and_obs_cnt_and_cloud_conf_and_advanced_features(X_train)
X_test = get_base_and_scl_and_obs_cnt_and_cloud_conf_and_advanced_features(X_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,...,ndvi,evi,evi2,ndre_b5,ndre_b6,ndre_b7,gndvi,ndwi,valid_observations,cloud_confidence
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608,...,0.218391,0.074277,-0.116667,0.081633,0.039216,0.009524,0.12192,-0.116667,7,0


In [23]:
train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:50:43,995] A new study created in memory with name: no-name-1caf9ee7-37c5-4529-ade2-79857c160792


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:50:53,209] Trial 0 finished with value: 0.9019608941866565 and parameters: {'n_estimators': 290, 'learning_rate': 0.06859241458017631, 'max_depth': 7, 'num_leaves': 199, 'subsample': 0.6348339773041654, 'colsample_bytree': 0.5254396031933607, 'min_child_samples': 34}. Best is trial 0 with value: 0.9019608941866565.
[I 2026-06-18 16:51:01,486] Trial 1 finished with value: 0.9001354859271548 and parameters: {'n_estimators': 316, 'learning_rate': 0.023972372204678016, 'max_depth': 5, 'num_leaves': 177, 'subsample': 0.9697291401148922, 'colsample_bytree': 0.8812290327205758, 'min_child_samples': 70}. Best is trial 0 with value: 0.9019608941866565.
[I 2026-06-18 16:51:26,152] Trial 2 finished with value: 0.9023879774107153 and parameters: {'n_estimators': 493, 'learning_rate': 0.012896099161238265, 'max_depth': 10, 'num_leaves': 77, 'subsample': 0.5102511026484333, 'colsample_bytree': 0.8063912238024218, 'min_child_samples': 91}. Best is trial 2 with value: 0.90238797741071

# Use neighbouring pixel values

In [24]:
def add_neighbouring_features(df):
    for suffix in ['_3x3_median', '_3x3_var', "_5x5_median", "_5x5_var"]:
        df[f"tci_{suffix}"] = compute_tci(df, suffix).copy()
        df[f"ndvi_{suffix}"] = compute_ndvi(df, suffix).copy()
        df[f"evi_{suffix}"] = compute_evi(df, suffix).copy()
        df[f"evi2_{suffix}"] = compute_evi2(df, suffix).copy()
        df[f"ndre_b7_{suffix}"] = compute_ndre_b7(df, suffix).copy()
        df[f"ndre_b6_{suffix}"] = compute_ndre_b6(df, suffix).copy()
        df[f"ndre_b5_{suffix}"] = compute_ndre_b5(df, suffix).copy()
        df[f"gndvi_{suffix}"] = compute_gndvi(df, suffix).copy()
        df[f"ndwi_{suffix}"] = compute_ndwi(df, suffix).copy()

add_neighbouring_features(Xy_train) 
add_neighbouring_features(Xy_test) 

neighbor_cols = [e for e in Xy_train.columns if e.endswith(("_median", '_var'))]
Xy_train[neighbor_cols].head(1)

,red_3x3_median,red_3x3_var,red_5x5_median,red_5x5_var,red_11x11_median,red_11x11_var,green_3x3_median,green_3x3_var,green_5x5_median,green_5x5_var,...,ndwi__5x5_median,tci__5x5_var,ndvi__5x5_var,evi__5x5_var,evi2__5x5_var,ndre_b7__5x5_var,ndre_b6__5x5_var,ndre_b5__5x5_var,gndvi__5x5_var,ndwi__5x5_var
0,0.133333,0.007843,0.133333,0.019608,0.12549,0.05098,0.094118,0.003922,0.094118,0.007843,...,-0.140496,-0.003529,-0.428571,-0.025467,-0.692308,-0.333333,-0.5,-0.666667,-0.027881,-0.692308


In [25]:
# Take into account this SCL
def get_base_and_scl_and_obs_cnt_and_cloud_conf_and_neighbor_advanced_features(X):
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl']
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    features += ['valid_observations']
    features += ['cloud_confidence']
    features += [col for col in X.columns if any(e in col for e in ('3x3', '5x5'))] 
    return X[features]

# Get train:test
X_train, y_train = Xy_train.drop(columns=['y']), Xy_train.y
X_test, y_test = Xy_test.drop(columns=['y']), Xy_test.y

# Get features
X_train = get_base_and_scl_and_obs_cnt_and_cloud_conf_and_neighbor_advanced_features(Xy_train)
X_test = get_base_and_scl_and_obs_cnt_and_cloud_conf_and_neighbor_advanced_features(Xy_test)

X_train.head(1)

,blue,green,red,rededge,B06,B07,nir,B8A,swir11,swir22,...,ndwi__5x5_median,tci__5x5_var,ndvi__5x5_var,evi__5x5_var,evi2__5x5_var,ndre_b7__5x5_var,ndre_b6__5x5_var,ndre_b5__5x5_var,gndvi__5x5_var,ndwi__5x5_var
0,0.066667,0.094118,0.133333,0.176471,0.192157,0.203922,0.207843,0.207843,0.262745,0.219608,...,-0.140496,-0.003529,-0.428571,-0.025467,-0.692308,-0.333333,-0.5,-0.666667,-0.027881,-0.692308


In [26]:
clf, yhat_train, yhat_test = train_test_lgbm(X_train, y_train, X_test, y_test, track_train_df.fold_idx);

[I 2026-06-18 16:51:33,173] A new study created in memory with name: no-name-ee80733e-c9e7-498a-bc6b-4e8395870cb4


Starting Optuna hyperparameter optimization...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2026-06-18 16:52:21,106] Trial 0 finished with value: 0.9114154457326562 and parameters: {'n_estimators': 218, 'learning_rate': 0.10501563590523513, 'max_depth': 9, 'num_leaves': 184, 'subsample': 0.832325547409436, 'colsample_bytree': 0.8436480686790615, 'min_child_samples': 79}. Best is trial 0 with value: 0.9114154457326562.
[I 2026-06-18 16:53:11,017] Trial 1 finished with value: 0.9118974861782538 and parameters: {'n_estimators': 144, 'learning_rate': 0.036978488543237584, 'max_depth': 15, 'num_leaves': 230, 'subsample': 0.7470778458776621, 'colsample_bytree': 0.7402780319571292, 'min_child_samples': 69}. Best is trial 1 with value: 0.9118974861782538.
[I 2026-06-18 16:53:47,133] Trial 2 finished with value: 0.9080705696593441 and parameters: {'n_estimators': 359, 'learning_rate': 0.019569357409414265, 'max_depth': 5, 'num_leaves': 208, 'subsample': 0.8608204533138153, 'colsample_bytree': 0.858427891484447, 'min_child_samples': 49}. Best is trial 1 with value: 0.911897486178253

In [30]:
# Dump to disk
import joblib

payload = {
    'X_train': X_train,
    'y_train': y_train,
    'track_train_df':  track_train_df,
    'X_test': X_test,
    'y_test': y_test,
    'track_test_df':  track_test_df,
    'clf': clf,
}

joblib.dump(payload, 'trained-2021.joblib')

['trained-2021.joblib']

# Use TCA
https://en.wikipedia.org/wiki/Tasseled_cap_transformation

# Use clustering min ?